# Section 3.3: Outlier Handling Pipeline

This notebook demonstrates a production-ready outlier detection and treatment pipeline with comprehensive audit logging.

**Objective:** Identify and treat outliers using statistical methods (IQR) combined with domain knowledge and visual diagnostics.

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Tuple, Any, List
import logging

# Configure logging for audit trail
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

## PHASE 0: CONFIGURATION

Define outlier detection and treatment strategies per feature.

In [ ]:
# ==============================================================================
# OUTLIER HANDLING CONFIGURATION
# ==============================================================================

OUTLIER_CONFIG = {
    'features': {
        'customer_age': {
            'detection_method': 'domain_based',          # 'iqr', 'zscore', 'isolation_forest', 'domain_based', or 'hybrid'
            'treat_method': 'remove',                     # 'remove', 'cap', 'transform', or 'flag'
            'lower_bound': 18,                            # Domain-based: remove if < 18 (legal lending age)
            'upper_bound': 100,                           # Domain-based: remove if > 100 (implausible age)
            'iqr_multiplier': 1.5,                        # For IQR method: 1.5 = standard Tukey fences
            'create_indicator': False,                    # Create binary outlier indicator
            'description': 'Age < 18 (legal threshold) or > 100 (implausible). Domain-based removal.'
        },
        'customer_income': {
            'detection_method': 'none',                   # Right-skew is natural; high income ≠ outlier
            'treat_method': 'transform',                  # Apply log transformation
            'transform_type': 'log1p',                    # log1p handles zeros gracefully
            'create_indicator': False,
            'description': 'Right-skewed distribution. Apply log transformation; retain all values.'
        },
        'employment_duration': {
            'detection_method': 'domain_based',
            'treat_method': 'remove',
            'upper_bound': 100,                           # Remove if > 100 years (implausible)
            'iqr_multiplier': 1.5,
            'create_indicator': False,
            'description': 'Employment duration > 100 years is implausible data entry error. Remove.'
        },
        'loan_amnt': {
            'detection_method': 'hybrid',                 # Combine IQR + distributional analysis
            'treat_method': 'remove',
            'upper_bound': 900000,                        # Domain + empirical: discontinuity in eCDF
            'iqr_multiplier': 1.5,
            'create_indicator': False,
            'description': 'Extreme values (1M, 3.5M) show eCDF discontinuity. Different lending process. Remove.'
        },
        'loan_int_rate': {
            'detection_method': 'iqr',
            'treat_method': 'retain',                     # Keep outliers; they represent legitimate high-risk loans
            'iqr_multiplier': 1.5,
            'create_indicator': False,
            'description': 'Outliers (rates >20%) are realistic. Represent legitimate high-risk loans. Retain.'
        },
        'cred_hist_length': {
            'detection_method': 'iqr',
            'treat_method': 'retain',                     # Keep all; outliers are natural (mirrors life expectancy)
            'iqr_multiplier': 1.5,
            'create_indicator': False,
            'description': 'Outliers reflect natural credit history distribution (life expectancy). Retain.'
        }
    },
    'analysis_before': True,                              # Analyze outliers before treating
    'fail_if_rows_dropped': False,                        # Don't fail on row removal; just log
}

## PHASE 1: OUTLIER DETECTION FUNCTIONS

In [ ]:
def analyze_outliers_all(
    df: pd.DataFrame,
    config: Dict[str, Any]
) -> Dict[str, Dict[str, Any]]:
    """
    Analyze outliers across all configured features.
    
    Analysis varies by detection method:
    - 'iqr': Report Tukey fence-based outliers
    - 'domain_based': Report values outside configured bounds
    - 'zscore': Report values with |z| > threshold
    - 'none': Skip analysis (no outliers to detect)
    - 'hybrid': Report both IQR and domain-based findings
    """
    analysis = {}
    
    logger.info("\n" + "="*70)
    logger.info("OUTLIER ANALYSIS (BEFORE TREATMENT)")
    logger.info("="*70)
    
    for feature_name in config['features'].keys():
        if feature_name not in df.columns:
            continue
        
        series = df[feature_name]
        if not pd.api.types.is_numeric_dtype(series):
            logger.warning(f"✗ {feature_name}: Not numeric; skipping")
            continue
        
        feature_config = config['features'][feature_name]
        detection = feature_config.get('detection_method')
        
        logger.info(f"\n{feature_name}:")
        logger.info(f"  Detection method: {detection}")
        
        # =====================================================================
        # ANALYSIS BY DETECTION METHOD
        # =====================================================================
        
        if detection == 'none':
            logger.info(f"  Status: No outlier detection (transformation only)")
            analysis[feature_name] = {'detection': 'none', 'count': 0}
        
        # =====================================================================
        # IQR-BASED ANALYSIS
        # =====================================================================
        elif detection == 'iqr':
            iqr_mult = feature_config.get('iqr_multiplier', 1.5)
            feature_analysis = analyze_outliers(series, feature_name, iqr_mult)
            analysis[feature_name] = feature_analysis
            
            logger.info(f"  IQR Multiplier: {iqr_mult}")
            logger.info(f"  Outliers detected: {feature_analysis['count']} ({feature_analysis['percent']:.3f}%)")
            logger.info(f"  Range: [{feature_analysis['min']:.2f}, {feature_analysis['max']:.2f}]")
            logger.info(f"  Q1={feature_analysis['q1']:.2f}, Q3={feature_analysis['q3']:.2f}, IQR={feature_analysis['iqr']:.2f}")
            logger.info(f"  IQR Fences: [{feature_analysis['lower_fence']:.2f}, {feature_analysis['upper_fence']:.2f}]")
        
        # =====================================================================
        # DOMAIN-BASED ANALYSIS
        # =====================================================================
        elif detection == 'domain_based':
            lower_bound = feature_config.get('lower_bound')
            upper_bound = feature_config.get('upper_bound')
            
            analysis[feature_name] = {
                'detection': 'domain_based',
                'lower_bound': lower_bound,
                'upper_bound': upper_bound,
                'min': float(series.min()),
                'max': float(series.max()),
                'mean': float(series.mean()),
                'median': float(series.median())
            }
            
            # Count how many would be removed
            if lower_bound is not None:
                below_lower = (series < lower_bound).sum()
                logger.info(f"  Lower bound: {feature_name} >= {lower_bound}")
                logger.info(f"    Values below bound: {below_lower}")
            
            if upper_bound is not None:
                above_upper = (series > upper_bound).sum()
                logger.info(f"  Upper bound: {feature_name} <= {upper_bound}")
                logger.info(f"    Values above bound: {above_upper}")
            
            logger.info(f"  Data range: [{series.min():.2f}, {series.max():.2f}]")
            logger.info(f"  Mean={series.mean():.2f}, Median={series.median():.2f}")
        
        # =====================================================================
        # Z-SCORE ANALYSIS
        # =====================================================================
        elif detection == 'zscore':
            threshold = feature_config.get('zscore_threshold', 3.0)
            z_scores = np.abs((series - series.mean()) / series.std())
            outliers_mask = z_scores > threshold
            n_outliers = outliers_mask.sum()
            pct_outliers = (n_outliers / len(series)) * 100
            
            analysis[feature_name] = {
                'detection': 'zscore',
                'threshold': threshold,
                'count': int(n_outliers),
                'percent': round(pct_outliers, 3),
                'min': float(series.min()),
                'max': float(series.max())
            }
            
            logger.info(f"  Z-score threshold: {threshold}")
            logger.info(f"  Outliers detected: {n_outliers} ({pct_outliers:.3f}%)")
        
        # =====================================================================
        # HYBRID ANALYSIS
        # =====================================================================
        elif detection == 'hybrid':
            # Run both IQR and domain-based analysis
            iqr_mult = feature_config.get('iqr_multiplier', 1.5)
            iqr_analysis = analyze_outliers(series, feature_name, iqr_mult)
            
            logger.info(f"  Method 1: IQR Analysis")
            logger.info(f"    Outliers: {iqr_analysis['count']} ({iqr_analysis['percent']:.3f}%)")
            logger.info(f"    IQR Fences: [{iqr_analysis['lower_fence']:.2f}, {iqr_analysis['upper_fence']:.2f}]")
            
            lower_bound = feature_config.get('lower_bound')
            upper_bound = feature_config.get('upper_bound')
            
            if lower_bound is not None or upper_bound is not None:
                logger.info(f"  Method 2: Domain-based Analysis")
                if lower_bound is not None:
                    below = (series < lower_bound).sum()
                    logger.info(f"    Below {lower_bound}: {below} values")
                if upper_bound is not None:
                    above = (series > upper_bound).sum()
                    logger.info(f"    Above {upper_bound}: {above} values")
            
            analysis[feature_name] = {
                'detection': 'hybrid',
                'iqr': iqr_analysis,
                'domain_bounds': {'lower': lower_bound, 'upper': upper_bound}
            }
    
    logger.info(f"\n" + "="*70)
    return analysis


## PHASE 2: OUTLIER TREATMENT FUNCTION

In [ ]:
def handle_outliers(
    df: pd.DataFrame,
    config: Dict[str, Any] = None,
    analysis: Dict[str, Dict[str, Any]] = None
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Treat outliers according to feature-specific strategies.
    
    Supports four treatment methods per feature:
    - 'remove': Delete rows containing outliers
    - 'cap': Replace outliers with fence values
    - 'transform': Apply mathematical transformation (log, sqrt, etc.)
    - 'retain': Keep outliers unchanged
    - 'flag': Create binary outlier indicator
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    config : Dict
        Outlier configuration
    analysis : Dict, optional
        Pre-computed outlier analysis (will compute if not provided)
    
    Returns:
    --------
    df_clean : pd.DataFrame
        Dataframe with outliers treated
    audit : Dict
        Comprehensive audit trail
    """
    if config is None:
        config = OUTLIER_CONFIG
    
    if analysis is None:
        analysis = analyze_outliers_all(df, config)
    
    audit = {
        'status': 'STARTED',
        'rows_input': len(df),
        'rows_output': len(df),
        'rows_removed': 0,
        'features_processed': [],
        'features_transformed': [],
        'details': {},
        'errors': [],
        'warnings': []
    }
    
    df_clean = df.copy()
    
    logger.info("\n" + "="*70)
    logger.info("OUTLIER TREATMENT EXECUTION")
    logger.info("="*70)
    
    # =========================================================================
    # PROCESS EACH FEATURE
    # =========================================================================
    for feature_name, feature_config in config['features'].items():
        if feature_name not in df_clean.columns:
            continue
        
        detection = feature_config.get('detection_method')
        treatment = feature_config.get('treat_method')
        
        logger.info(f"\n{'─'*70}")
        logger.info(f"Feature: {feature_name}")
        logger.info(f"  Detection: {detection}")
        logger.info(f"  Treatment: {treatment}")
        
        detail = {
            'detection_method': detection,
            'treat_method': treatment,
            'outliers_detected': 0,
            'rows_removed': 0,
            'values_capped': 0,
            'values_transformed': 0
        }
        
        # =====================================================================
        # TREATMENT 1: REMOVE ROWS
        # =====================================================================
        if treatment == 'remove':
            # Apply domain-based bounds if configured
            rows_before = len(df_clean)
            
            if 'lower_bound' in feature_config:
                lower = feature_config['lower_bound']
                mask = df_clean[feature_name] >= lower
                logger.info(f"  Lower bound removal: {feature_name} >= {lower}")
                df_clean = df_clean[mask]
            
            if 'upper_bound' in feature_config:
                upper = feature_config['upper_bound']
                mask = df_clean[feature_name] <= upper
                logger.info(f"  Upper bound removal: {feature_name} <= {upper}")
                df_clean = df_clean[mask]
            
            rows_removed = rows_before - len(df_clean)
            detail['rows_removed'] = int(rows_removed)
            audit['rows_removed'] += rows_removed
            
            logger.info(f"  Rows removed: {rows_removed}")
        
        # =====================================================================
        # TREATMENT 2: CAP OUTLIERS
        # =====================================================================
        elif treatment == 'cap':
            if feature_name in analysis:
                feat_analysis = analysis[feature_name]
                lower_fence = feat_analysis['lower_fence']
                upper_fence = feat_analysis['upper_fence']
                
                # Cap lower values
                mask_lower = df_clean[feature_name] < lower_fence
                n_capped_lower = mask_lower.sum()
                df_clean.loc[mask_lower, feature_name] = lower_fence
                
                # Cap upper values
                mask_upper = df_clean[feature_name] > upper_fence
                n_capped_upper = mask_upper.sum()
                df_clean.loc[mask_upper, feature_name] = upper_fence
                
                n_capped = n_capped_lower + n_capped_upper
                detail['values_capped'] = int(n_capped)
                logger.info(f"  Values capped: {n_capped} (lower: {n_capped_lower}, upper: {n_capped_upper})")
        
        # =====================================================================
        # TREATMENT 3: TRANSFORM
        # =====================================================================
        elif treatment == 'transform':
            transform_type = feature_config.get('transform_type', 'log1p')
            new_col_name = f"{feature_name}_transformed"
            
            if transform_type == 'log1p':
                df_clean[new_col_name] = np.log1p(df_clean[feature_name])
                logger.info(f"  Applied log1p transformation → '{new_col_name}'")
                audit['features_transformed'].append(new_col_name)
            
            elif transform_type == 'sqrt':
                df_clean[new_col_name] = np.sqrt(df_clean[feature_name])
                logger.info(f"  Applied sqrt transformation → '{new_col_name}'")
                audit['features_transformed'].append(new_col_name)
            
            elif transform_type == 'yeo-johnson':
                # Power transformation to achieve normality
                from sklearn.preprocessing import PowerTransformer
                pt = PowerTransformer(method='yeo-johnson')
                df_clean[new_col_name] = pt.fit_transform(df_clean[[feature_name]])
                logger.info(f"  Applied Yeo-Johnson transformation → '{new_col_name}'")
                audit['features_transformed'].append(new_col_name)
            
            detail['values_transformed'] = len(df_clean)
        
        # =====================================================================
        # TREATMENT 4: RETAIN
        # =====================================================================
        elif treatment == 'retain':
            logger.info(f"  Outliers retained as legitimate values")
        
        audit['features_processed'].append(feature_name)
        audit['details'][feature_name] = detail
    
    # =========================================================================
    # UPDATE COUNTS
    # =========================================================================
    audit['rows_output'] = len(df_clean)
    
    # =========================================================================
    # SUMMARY
    # =========================================================================
    logger.info(f"\n" + "="*70)
    logger.info("OUTLIER HANDLING SUMMARY")
    logger.info("="*70)
    logger.info(f"Status: SUCCESS")
    logger.info(f"Features processed: {len(audit['features_processed'])}")
    logger.info(f"Features transformed: {len(audit['features_transformed'])}")
    if audit['features_transformed']:
        logger.info(f"  → {', '.join(audit['features_transformed'])}")
    logger.info(f"Rows: {audit['rows_input']} → {audit['rows_output']} (removed: {audit['rows_removed']})")
    logger.info(f"="*70)
    
    audit['status'] = 'SUCCESS'
    return df_clean, audit

## PHASE 3: CREATE SAMPLE DATA

In [ ]:
# Create sample data with outliers matching your patterns
np.random.seed(42)
n_rows = 10000

sample_data = pd.DataFrame({
    'customer_age': np.concatenate([
        np.random.normal(35, 15, n_rows - 50),  # Normal distribution
        np.array([5, 10, 15, 125, 130, 140, 150, 160, 180])  # Outliers: below 18 and above 100
    ])[:n_rows].astype(int),
    'customer_income': np.random.exponential(30000, n_rows),  # Right-skewed
    'employment_duration': np.concatenate([
        np.random.uniform(0, 50, n_rows - 5),
        np.array([105, 110, 123, 130, 140])  # Outliers: > 100 years
    ])[:n_rows],
    'loan_amnt': np.concatenate([
        np.random.exponential(25000, n_rows - 3),
        np.array([1000000, 1500000, 3500000])  # Extreme outliers
    ])[:n_rows],
    'loan_int_rate': np.random.uniform(2, 25, n_rows),  # Some > 20%
    'cred_hist_length': np.random.normal(10, 8, n_rows)  # Natural outliers at extremes
})

print("Sample data created")
print(f"Shape: {sample_data.shape}")

## PHASE 4: RUN ANALYSIS

In [ ]:
# Analyze outliers BEFORE treatment
outlier_analysis = analyze_outliers_all(sample_data, OUTLIER_CONFIG)

## PHASE 5: RUN TREATMENT

In [ ]:
# Execute outlier treatment
data_clean, outlier_audit = handle_outliers(sample_data, OUTLIER_CONFIG, outlier_analysis)

## PHASE 6: INSPECT RESULTS

In [ ]:
print("\n" + "="*70)
print("OUTLIER TREATMENT RESULTS")
print("="*70)
print(f"\nStatus: {outlier_audit['status']}")
print(f"\nRows:")
print(f"  Input:   {outlier_audit['rows_input']}")
print(f"  Output:  {outlier_audit['rows_output']}")
print(f"  Removed: {outlier_audit['rows_removed']}")

print(f"\nFeatures processed: {len(outlier_audit['features_processed'])}")
for feature in outlier_audit['features_processed']:
    details = outlier_audit['details'][feature]
    print(f"\n  {feature}:")
    print(f"    Method: {details['treat_method']}")
    if details['rows_removed'] > 0:
        print(f"    Rows removed: {details['rows_removed']}")
    if details['values_capped'] > 0:
        print(f"    Values capped: {details['values_capped']}")
    if details['values_transformed'] > 0:
        print(f"    Values transformed: {details['values_transformed']}")

if outlier_audit['features_transformed']:
    print(f"\nNew transformed features:")
    for feat in outlier_audit['features_transformed']:
        print(f"  - {feat}")

print(f"\n" + "="*70)

## PHASE 7: INTEGRATION INTO FULL PIPELINE

In [ ]:
class DataCleaningPipeline:
    """
    Full data cleaning pipeline with duplicates, missing values, and outliers.
    """
    
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.audit_trail = {}
        self.analysis = {}
    
    def execute(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
        """
        Execute full cleaning pipeline.
        """
        df_current = df.copy()
        
        # Step 1: Duplicates
        logger.info("\n" + "#"*70)
        logger.info("# STEP 1: DUPLICATE HANDLING")
        logger.info("#"*70)
        # df_current, audit = handle_duplicates(...)
        # self.audit_trail['duplicates'] = audit
        
        # Step 2: Missing values
        logger.info("\n" + "#"*70)
        logger.info("# STEP 2: MISSING VALUE HANDLING")
        logger.info("#"*70)
        # df_current, audit = handle_missing_values(...)
        # self.audit_trail['missing_values'] = audit
        
        # Step 3: Outliers
        logger.info("\n" + "#"*70)
        logger.info("# STEP 3: OUTLIER HANDLING")
        logger.info("#"*70)
        analysis = analyze_outliers_all(df_current, self.config['outlier_handling'])
        self.analysis['outliers'] = analysis
        
        df_current, audit = handle_outliers(
            df_current,
            self.config['outlier_handling'],
            analysis=analysis
        )
        self.audit_trail['outliers'] = audit
        
        return df_current, self.audit_trail

## PHASE 8: FULL PIPELINE EXECUTION

In [ ]:
# Full pipeline config
FULL_PIPELINE_CONFIG = {
    'outlier_handling': OUTLIER_CONFIG,
}

# Execute
pipeline = DataCleaningPipeline(FULL_PIPELINE_CONFIG)
data_final, audit_trail = pipeline.execute(sample_data)

print("\n" + "="*70)
print("FINAL PIPELINE SUMMARY")
print("="*70)
print(f"Input rows: {sample_data.shape[0]}")
print(f"Output rows: {data_final.shape[0]}")
print(f"Rows removed: {sample_data.shape[0] - data_final.shape[0]}")
print(f"Input columns: {sample_data.shape[1]}")
print(f"Output columns: {data_final.shape[1]}")
print(f"\n✓ Cleaning complete")
print("="*70)